# Mandatory Practical Work 01 - CNNs on iCoSimal V3 Dataset

**MSE FTP Deep Learning**

---

## Group Members
- Student 1: Marcos, Costa
- Student 2: Jose Pablo, Munoz
- Student 3: Artemii, Ponomarenko

---

## Table of Contents

1. [Setup and Data Loading](#1-setup)
2. [Experiment A: CNN Depth Progression](#2-depth) - *Mandatory (a)*
3. [Experiment B: Hyperparameter Tuning](#3-hyperparams) - *Mandatory (b)*
4. [Experiment C: Overfitting vs Underfitting](#4-fitting) - *Mandatory (c)*
5. [Experiment D: Regularization Techniques](#5-regularization) - *Mandatory (d)*
6. [Experiment E: Optimizer Comparison](#6-optimizers) - *Mandatory (e)*
7. [Optional F: Error Analysis](#7-error-analysis) - *Optional*
8. [Optional G: ResNet Comparison](#8-resnet) - *Optional*
9. [Optional H: Data Augmentation](#9-augmentation) - *Optional*
10. [Results Summary and Conclusions](#10-conclusions)

---
## 1. Setup and Data Loading <a id='1-setup'></a>

### Step 1: Install dependencies (run once in terminal)
```bash
pip install torch torchvision matplotlib wandb tqdm
```

### Step 2: Setup Weights & Biases
```bash
wandb login
```
Follow the prompts to authenticate with your W&B account.

### Step 3: Download the dataset
Download the iCoSimal V3 dataset from the course link and extract it.

**Expected structure:**
```
data/
├── train/
│   ├── cat/
│   ├── chicken/
│   └── ... (10 classes)
└── validate/
    ├── cat/
    └── ...
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import os
import wandb

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ============================================================
# CONFIGURATION - Modify these paths and settings as needed
# ============================================================

# Dataset paths - UPDATE THESE to match your setup
DATA_ROOT = "./data"  # Change to your dataset location
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR = os.path.join(DATA_ROOT, "validate")

# Image size - Use 64x64 for fast experiments, 224x224 for final results
IMG_SIZE = 64  # Start small for quick iteration

# Training defaults
BATCH_SIZE = 64
NUM_WORKERS = 4  # Reduce to 0 if you get multiprocessing errors
NUM_CLASSES = 10

# W&B Configuration
WANDB_PROJECT = "icosimal-cnn"  # Your W&B project name
WANDB_ENTITY = None  # Set to your W&B username/team, or None for default

# Class names for reference
CLASS_NAMES = ['cat', 'chicken', 'cow', 'dog', 'elephant', 
               'horse', 'rabbit', 'sheep', 'squirrel', 'zebra']

In [ ]:
def get_data_loaders(img_size=IMG_SIZE, batch_size=BATCH_SIZE, augment=False):
    """Create train and validation data loaders."""
    
    # Base transforms
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
    
    if augment:
        train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            normalize
        ])
    else:
        train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            normalize
        ])
    
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        normalize
    ])
    
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                              shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, 
                            shuffle=False, num_workers=NUM_WORKERS)
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    
    return train_loader, val_loader, train_dataset, val_dataset

In [ ]:
# Test data loading
train_loader, val_loader, train_dataset, val_dataset = get_data_loaders()

# Visualize some samples
def show_samples(dataset, n=8):
    fig, axes = plt.subplots(1, n, figsize=(16, 2))
    for i in range(n):
        idx = np.random.randint(len(dataset))
        img, label = dataset[idx]
        # Denormalize for display
        img = img.permute(1, 2, 0).numpy()
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)
        axes[i].imshow(img)
        axes[i].set_title(CLASS_NAMES[label])
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

show_samples(train_dataset)

---
## Training Utilities with W&B Integration

Helper functions used throughout all experiments. All training is automatically logged to Weights & Biases.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / total, 100. * correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / total, 100. * correct / total


def train_model(model, train_loader, val_loader, config, device=device):
    """
    Full training loop with W&B logging.
    
    config dict should contain:
        - experiment: experiment name (e.g., 'A_depth')
        - model_name: model architecture name
        - epochs: number of epochs
        - lr: learning rate
        - optimizer: 'adam', 'sgd', 'sgd_momentum', 'rmsprop'
        - weight_decay: L2 regularization (default 0)
        - batch_size: batch size
        - img_size: image size
        - (optional) dropout_rate, augmentation, etc.
    """
    # Initialize W&B run
    run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=f"{config.get('experiment', 'exp')}_{config.get('model_name', 'model')}",
        config=config,
        reinit=True
    )
    
    criterion = nn.CrossEntropyLoss()
    
    # Setup optimizer based on config
    opt_name = config.get('optimizer', 'adam').lower()
    lr = config.get('lr', 0.001)
    weight_decay = config.get('weight_decay', 0)
    
    if opt_name == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif opt_name == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif opt_name == 'sgd_momentum':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    elif opt_name == 'rmsprop':
        optimizer = optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    model = model.to(device)
    epochs = config.get('epochs', 10)
    
    # Log model architecture
    wandb.watch(model, log='all', log_freq=100)
    
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        
        # Log metrics to W&B
        wandb.log({
            'epoch': epoch + 1,
            'train/loss': train_loss,
            'train/accuracy': train_acc,
            'val/loss': val_loss,
            'val/accuracy': val_acc,
            'train_val_gap': train_acc - val_acc,
            'learning_rate': optimizer.param_groups[0]['lr']
        })
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")
    
    # Log final summary metrics
    wandb.summary['best_val_accuracy'] = max(history['val_acc'])
    wandb.summary['final_val_accuracy'] = history['val_acc'][-1]
    wandb.summary['final_train_accuracy'] = history['train_acc'][-1]
    wandb.summary['final_gap'] = history['train_acc'][-1] - history['val_acc'][-1]
    
    # Finish the W&B run
    wandb.finish()
    
    return history


def plot_history(history, title="Training History"):
    """Plot training curves locally."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(history['train_loss'], label='Train')
    ax1.plot(history['val_loss'], label='Validation')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Loss')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(history['train_acc'], label='Train')
    ax2.plot(history['val_acc'], label='Validation')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Accuracy')
    ax2.legend()
    ax2.grid(True)
    
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def compare_histories(histories, labels, title="Model Comparison"):
    """Compare multiple training runs locally."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    for hist, label in zip(histories, labels):
        ax1.plot(hist['val_loss'], label=label)
        ax2.plot(hist['val_acc'], label=label)
    
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Validation Loss')
    ax1.set_title('Validation Loss Comparison')
    ax1.legend()
    ax1.grid(True)
    
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Validation Accuracy (%)')
    ax2.set_title('Validation Accuracy Comparison')
    ax2.legend()
    ax2.grid(True)
    
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

---
## Model Architectures

All CNN architectures used in experiments.

In [ ]:
class ShallowCNN(nn.Module):
    """1 convolutional layer - Very simple baseline."""
    def __init__(self, num_classes=10, img_size=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        feat_size = img_size // 2
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * feat_size * feat_size, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))


class MediumCNN(nn.Module):
    """3 convolutional layers - Moderate depth."""
    def __init__(self, num_classes=10, img_size=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        feat_size = img_size // 8
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * feat_size * feat_size, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))


class DeepCNN(nn.Module):
    """5 convolutional layers with BatchNorm."""
    def __init__(self, num_classes=10, img_size=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))


class TinyCNN(nn.Module):
    """Extremely limited capacity - will underfit."""
    def __init__(self, num_classes=10, img_size=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 4, kernel_size=5, stride=2),
            nn.ReLU(),
        )
        feat_size = (img_size - 5) // 2 + 1
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(4 * feat_size * feat_size, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))


class LargeCNN(nn.Module):
    """High capacity model - will overfit on small data."""
    def __init__(self, num_classes=10, img_size=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))


class RegularizedCNN(nn.Module):
    """CNN with configurable dropout."""
    def __init__(self, num_classes=10, img_size=64, dropout_rate=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(dropout_rate),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(dropout_rate),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(dropout_rate),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))

---
## 2. Experiment A: CNN Depth Progression <a id='2-depth'></a>

**Objective (a):** Start from simple CNN architectures and progressively increase their complexity to show the benefit of depth.

### Experiment Plan
1. **Shallow CNN** (1 conv layer) - Baseline
2. **Medium CNN** (3 conv layers) - More feature extraction
3. **Deep CNN** (5 conv layers) - Full hierarchical features

All runs will be logged to W&B for comparison.

In [ ]:
# Run Experiment A: Depth Comparison
print("="*60)
print("EXPERIMENT A: CNN DEPTH PROGRESSION")
print("="*60)

train_loader, val_loader, _, _ = get_data_loaders(img_size=IMG_SIZE)

depth_results = {}
depth_histories = []
depth_labels = []

models_depth = [
    ("Shallow_1conv", ShallowCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE)),
    ("Medium_3conv", MediumCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE)),
    ("Deep_5conv", DeepCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE)),
]

for name, model in models_depth:
    print(f"\n--- Training {name} ---")
    
    config = {
        'experiment': 'A_depth',
        'model_name': name,
        'epochs': 15,
        'lr': 0.001,
        'optimizer': 'adam',
        'batch_size': BATCH_SIZE,
        'img_size': IMG_SIZE,
        'num_conv_layers': int(name.split('_')[1][0]) if name[0] != 'S' else 1
    }
    
    history = train_model(model, train_loader, val_loader, config)
    
    depth_results[name] = {
        'final_train_acc': history['train_acc'][-1],
        'final_val_acc': history['val_acc'][-1],
        'best_val_acc': max(history['val_acc'])
    }
    depth_histories.append(history)
    depth_labels.append(name)

In [ ]:
# Visualize depth comparison
compare_histories(depth_histories, depth_labels, "Experiment A: CNN Depth Comparison")

print("\n" + "="*60)
print("EXPERIMENT A: SUMMARY")
print("="*60)
for name, results in depth_results.items():
    print(f"{name:20} | Best Val Acc: {results['best_val_acc']:.2f}%")

print("\n[View detailed metrics and comparisons on W&B dashboard]")

### Experiment A: Analysis

**YOUR OBSERVATIONS HERE:**

- Shallow CNN: [Describe performance - limited feature extraction capability]
- Medium CNN: [Describe improvement - better at capturing patterns]
- Deep CNN: [Describe - hierarchical features, but may need more data/regularization]

**Key Insight:** Deeper networks can learn more complex hierarchical features, but...

---
## 3. Experiment B: Hyperparameter Tuning <a id='3-hyperparams'></a>

**Objective (b):** Show the importance of hyperparameter tuning.

### Experiment Plan
Using the Medium CNN, we'll vary:
1. **Learning Rate**: 0.1, 0.01, 0.001, 0.0001
2. **Batch Size**: 16, 32, 64, 128

In [ ]:
# Experiment B.1: Learning Rate Sweep
print("="*60)
print("EXPERIMENT B.1: LEARNING RATE COMPARISON")
print("="*60)

learning_rates = [0.1, 0.01, 0.001, 0.0001]

lr_histories = []
lr_labels = []
lr_results = {}

train_loader, val_loader, _, _ = get_data_loaders(img_size=IMG_SIZE, batch_size=64)

for lr in learning_rates:
    print(f"\n--- Learning Rate: {lr} ---")
    model = MediumCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE)
    
    config = {
        'experiment': 'B1_learning_rate',
        'model_name': f'MediumCNN_lr{lr}',
        'epochs': 10,
        'lr': lr,
        'optimizer': 'adam',
        'batch_size': 64,
        'img_size': IMG_SIZE
    }
    
    history = train_model(model, train_loader, val_loader, config)
    
    lr_histories.append(history)
    lr_labels.append(f"LR={lr}")
    lr_results[lr] = max(history['val_acc'])

In [ ]:
compare_histories(lr_histories, lr_labels, "Experiment B.1: Learning Rate Comparison")

print("\nBest validation accuracy per learning rate:")
for lr, acc in lr_results.items():
    print(f"  LR={lr}: {acc:.2f}%")

best_lr = max(lr_results, key=lr_results.get)
print(f"\nBest LR: {best_lr}")

In [ ]:
# Experiment B.2: Batch Size Sweep
print("="*60)
print("EXPERIMENT B.2: BATCH SIZE COMPARISON")
print("="*60)

batch_sizes = [16, 32, 64, 128]

bs_histories = []
bs_labels = []
bs_results = {}

for bs in batch_sizes:
    print(f"\n--- Batch Size: {bs} ---")
    train_loader, val_loader, _, _ = get_data_loaders(img_size=IMG_SIZE, batch_size=bs)
    
    model = MediumCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE)
    
    config = {
        'experiment': 'B2_batch_size',
        'model_name': f'MediumCNN_bs{bs}',
        'epochs': 10,
        'lr': best_lr,
        'optimizer': 'adam',
        'batch_size': bs,
        'img_size': IMG_SIZE
    }
    
    history = train_model(model, train_loader, val_loader, config)
    
    bs_histories.append(history)
    bs_labels.append(f"BS={bs}")
    bs_results[bs] = max(history['val_acc'])

In [ ]:
compare_histories(bs_histories, bs_labels, "Experiment B.2: Batch Size Comparison")

print("\nBest validation accuracy per batch size:")
for bs, acc in bs_results.items():
    print(f"  BS={bs}: {acc:.2f}%")

### Experiment B: Analysis

**YOUR OBSERVATIONS HERE:**

**Learning Rate Impact:**
- Too high (0.1): [oscillation/divergence]
- Too low (0.0001): [slow convergence]
- Optimal: [describe]

**Batch Size Impact:**
- Small batches: [noisier gradients, potential regularization effect]
- Large batches: [smoother gradients, may need LR adjustment]

**Key Insight:** Hyperparameters significantly affect training...

---
## 4. Experiment C: Overfitting vs Underfitting <a id='4-fitting'></a>

**Objective (c):** Show models that overfit and underfit and explain the reasons.

### Experiment Plan
1. **Underfitting**: Very simple model (tiny capacity)
2. **Overfitting**: Complex model on small subset of data
3. **Good Fit**: Balanced model with appropriate capacity

In [ ]:
# Experiment C: Fitting Comparison
print("="*60)
print("EXPERIMENT C: OVERFITTING vs UNDERFITTING")
print("="*60)

# Full data loaders
train_loader_full, val_loader, train_dataset, _ = get_data_loaders(img_size=IMG_SIZE)

# Small subset for overfitting demo (only 500 samples)
small_indices = np.random.choice(len(train_dataset), size=500, replace=False)
small_dataset = Subset(train_dataset, small_indices)
train_loader_small = DataLoader(small_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS)

fit_histories = []
fit_labels = []

In [ ]:
# 1. Underfitting: Tiny model on full data
print("\n--- Underfitting Demo: Tiny model, full data ---")
model_underfit = TinyCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE)

config = {
    'experiment': 'C_fitting',
    'model_name': 'TinyCNN_underfit',
    'epochs': 30,
    'lr': 0.001,
    'optimizer': 'adam',
    'batch_size': BATCH_SIZE,
    'img_size': IMG_SIZE,
    'scenario': 'underfit',
    'train_samples': 'full'
}

hist_underfit = train_model(model_underfit, train_loader_full, val_loader, config)
fit_histories.append(hist_underfit)
fit_labels.append("Underfit (tiny model)")

In [ ]:
# 2. Overfitting: Large model on small data
print("\n--- Overfitting Demo: Large model, small data (500 samples) ---")
model_overfit = LargeCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE)

config = {
    'experiment': 'C_fitting',
    'model_name': 'LargeCNN_overfit',
    'epochs': 30,
    'lr': 0.001,
    'optimizer': 'adam',
    'batch_size': 32,
    'img_size': IMG_SIZE,
    'scenario': 'overfit',
    'train_samples': 500
}

hist_overfit = train_model(model_overfit, train_loader_small, val_loader, config)
fit_histories.append(hist_overfit)
fit_labels.append("Overfit (large model, small data)")

In [ ]:
# 3. Good fit: Medium model on full data
print("\n--- Good Fit: Medium model, full data ---")
model_goodfit = MediumCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE)

config = {
    'experiment': 'C_fitting',
    'model_name': 'MediumCNN_goodfit',
    'epochs': 30,
    'lr': 0.001,
    'optimizer': 'adam',
    'batch_size': BATCH_SIZE,
    'img_size': IMG_SIZE,
    'scenario': 'good_fit',
    'train_samples': 'full'
}

hist_goodfit = train_model(model_goodfit, train_loader_full, val_loader, config)
fit_histories.append(hist_goodfit)
fit_labels.append("Good Fit (medium model)")

In [ ]:
# Plot individual histories to see train vs val gap
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (hist, label) in enumerate(zip(fit_histories, fit_labels)):
    axes[i].plot(hist['train_acc'], label='Train')
    axes[i].plot(hist['val_acc'], label='Validation')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Accuracy (%)')
    axes[i].set_title(label)
    axes[i].legend()
    axes[i].grid(True)
    
    gap = hist['train_acc'][-1] - hist['val_acc'][-1]
    axes[i].text(0.5, 0.1, f'Gap: {gap:.1f}%', transform=axes[i].transAxes)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("EXPERIMENT C: SUMMARY")
print("="*60)
for hist, label in zip(fit_histories, fit_labels):
    train_final = hist['train_acc'][-1]
    val_final = hist['val_acc'][-1]
    print(f"{label:35} | Train: {train_final:.1f}% | Val: {val_final:.1f}% | Gap: {train_final-val_final:.1f}%")

### Experiment C: Analysis

**YOUR OBSERVATIONS HERE:**

**Underfitting (Tiny Model):**
- Both train and val accuracy are low
- Small gap between train and val
- Reason: Model lacks capacity to learn the patterns

**Overfitting (Large Model, Small Data):**
- Training accuracy approaches 100%
- Large gap between train and val
- Reason: Model memorizes training data instead of learning general patterns

**Good Fit:**
- Reasonable train accuracy
- Moderate gap between train and val
- Reason: Appropriate capacity for the data

---
## 5. Experiment D: Regularization Techniques <a id='5-regularization'></a>

**Objective (d):** Experiment with regularization techniques.

### Experiment Plan
Test regularization on limited data:
1. **No regularization** (baseline)
2. **Dropout**
3. **L2 Weight Decay**
4. **Dropout + Weight Decay**

In [ ]:
# Experiment D: Regularization
print("="*60)
print("EXPERIMENT D: REGULARIZATION TECHNIQUES")
print("="*60)

# Use smaller dataset to show regularization effect
small_indices = np.random.choice(len(train_dataset), size=1000, replace=False)
small_dataset = Subset(train_dataset, small_indices)
train_loader_reg = DataLoader(small_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS)

reg_configs = [
    ("No_Regularization", 0.0, 0.0),
    ("Dropout_0.3", 0.3, 0.0),
    ("WeightDecay_1e-4", 0.0, 1e-4),
    ("Dropout_WD_combined", 0.3, 1e-4),
]

reg_histories = []
reg_labels = []

In [ ]:
for name, dropout, weight_decay in reg_configs:
    print(f"\n--- {name} ---")
    model = RegularizedCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE, dropout_rate=dropout)
    
    config = {
        'experiment': 'D_regularization',
        'model_name': name,
        'epochs': 30,
        'lr': 0.001,
        'optimizer': 'adam',
        'weight_decay': weight_decay,
        'dropout_rate': dropout,
        'batch_size': 32,
        'img_size': IMG_SIZE,
        'train_samples': 1000
    }
    
    history = train_model(model, train_loader_reg, val_loader, config)
    
    reg_histories.append(history)
    reg_labels.append(name.replace('_', ' '))

In [ ]:
# Plot regularization comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (hist, label) in enumerate(zip(reg_histories, reg_labels)):
    axes[i].plot(hist['train_acc'], label='Train')
    axes[i].plot(hist['val_acc'], label='Validation')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Accuracy (%)')
    axes[i].set_title(label)
    axes[i].legend()
    axes[i].grid(True)

plt.tight_layout()
plt.show()

compare_histories(reg_histories, reg_labels, "Experiment D: Regularization Comparison")

print("\n" + "="*60)
print("EXPERIMENT D: SUMMARY")
print("="*60)
for hist, label in zip(reg_histories, reg_labels):
    best_val = max(hist['val_acc'])
    final_gap = hist['train_acc'][-1] - hist['val_acc'][-1]
    print(f"{label:30} | Best Val: {best_val:.1f}% | Final Gap: {final_gap:.1f}%")

### Experiment D: Analysis

**YOUR OBSERVATIONS HERE:**

- No regularization: [largest gap, overfitting]
- Dropout: [reduces gap by randomly dropping neurons]
- Weight decay: [penalizes large weights, smoother decision boundary]
- Combined: [best generalization]

**Key Insight:** Regularization techniques help bridge the generalization gap...

---
## 6. Experiment E: Optimizer Comparison <a id='6-optimizers'></a>

**Objective (e):** Experiment with different optimization algorithms.

### Experiment Plan
Compare optimizers on the same architecture:
1. **SGD** (vanilla)
2. **SGD + Momentum**
3. **RMSprop**
4. **Adam**

In [ ]:
# Experiment E: Optimizer Comparison
print("="*60)
print("EXPERIMENT E: OPTIMIZER COMPARISON")
print("="*60)

train_loader, val_loader, _, _ = get_data_loaders(img_size=IMG_SIZE, batch_size=64)

optimizer_configs = [
    ('SGD', 'sgd', 0.01),
    ('SGD_Momentum', 'sgd_momentum', 0.01),
    ('RMSprop', 'rmsprop', 0.001),
    ('Adam', 'adam', 0.001),
]

opt_histories = []
opt_labels = []

In [ ]:
for name, opt_type, lr in optimizer_configs:
    print(f"\n--- {name} ---")
    model = MediumCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE)
    
    config = {
        'experiment': 'E_optimizers',
        'model_name': f'MediumCNN_{name}',
        'epochs': 20,
        'lr': lr,
        'optimizer': opt_type,
        'batch_size': 64,
        'img_size': IMG_SIZE
    }
    
    history = train_model(model, train_loader, val_loader, config)
    
    opt_histories.append(history)
    opt_labels.append(name)

In [ ]:
compare_histories(opt_histories, opt_labels, "Experiment E: Optimizer Comparison")

print("\n" + "="*60)
print("EXPERIMENT E: SUMMARY")
print("="*60)
for hist, label in zip(opt_histories, opt_labels):
    best_val = max(hist['val_acc'])
    print(f"{label:20} | Best Val Acc: {best_val:.2f}%")

### Experiment E: Analysis

**YOUR OBSERVATIONS HERE:**

- **SGD**: [slow convergence, sensitive to LR]
- **SGD + Momentum**: [faster convergence, helps escape local minima]
- **RMSprop**: [adaptive LR, good for non-stationary problems]
- **Adam**: [combines momentum + adaptive LR, generally robust]

**Key Insight:** Adaptive optimizers like Adam often converge faster and are less sensitive to hyperparameters...

---
## 7. Optional: Data Augmentation <a id='7-augmentation'></a>

**Optional Objective:** Experiment with data augmentation techniques to improve model generalization.

### Experiment Plan
1. Train without augmentation
2. Train with augmentation (flip, rotation, color jitter)

In [ ]:
# Experiment: Data Augmentation
print("="*60)
print("OPTIONAL: DATA AUGMENTATION")
print("="*60)

aug_histories = []
aug_labels = []

# Without augmentation
print("\n--- Without Augmentation ---")
train_loader_no_aug, val_loader, _, _ = get_data_loaders(img_size=IMG_SIZE, augment=False)
model = DeepCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE)

config = {
    'experiment': 'Optional_augmentation',
    'model_name': 'DeepCNN_no_aug',
    'epochs': 25,
    'lr': 0.001,
    'optimizer': 'adam',
    'batch_size': BATCH_SIZE,
    'img_size': IMG_SIZE,
    'augmentation': False
}

history = train_model(model, train_loader_no_aug, val_loader, config)
aug_histories.append(history)
aug_labels.append("No Augmentation")

In [ ]:
# With augmentation
print("\n--- With Augmentation ---")
train_loader_aug, val_loader, _, _ = get_data_loaders(img_size=IMG_SIZE, augment=True)
model = DeepCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE)

config = {
    'experiment': 'Optional_augmentation',
    'model_name': 'DeepCNN_with_aug',
    'epochs': 25,
    'lr': 0.001,
    'optimizer': 'adam',
    'batch_size': BATCH_SIZE,
    'img_size': IMG_SIZE,
    'augmentation': True,
    'aug_transforms': ['horizontal_flip', 'rotation_15', 'color_jitter']
}

history = train_model(model, train_loader_aug, val_loader, config)
aug_histories.append(history)
aug_labels.append("With Augmentation")

---
## 7. Optional F: Error Analysis <a id='7-error-analysis'></a>

**Optional Objective:** Perform error analysis to identify common misclassifications.

### Experiment Plan
1. Train a good model (DeepCNN)
2. Get predictions on validation set
3. Analyze confusion matrix
4. Visualize misclassified samples

In [ ]:
# Error Analysis Functions
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

def get_predictions(model, loader, device):
    """Get all predictions and labels from a data loader."""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = outputs.max(1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


def plot_confusion_matrix(y_true, y_pred, class_names):
    """Plot confusion matrix as a heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()
    return cm


def get_misclassified_samples(dataset, preds, labels, n=5):
    """Get indices and info of misclassified samples."""
    misclassified = []
    for idx in range(len(preds)):
        if preds[idx] != labels[idx]:
            misclassified.append({
                'idx': idx,
                'true_label': labels[idx],
                'pred_label': preds[idx]
            })
    return misclassified[:n*10]  # Return more than needed for variety


def show_misclassified(dataset, misclassified_info, class_names, n=10):
    """Visualize misclassified samples."""
    n = min(n, len(misclassified_info))
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()
    
    for i, info in enumerate(misclassified_info[:n]):
        img, _ = dataset[info['idx']]
        img = img.permute(1, 2, 0).numpy()
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)
        
        axes[i].imshow(img)
        axes[i].set_title(f"True: {class_names[info['true_label']]}\nPred: {class_names[info['pred_label']]}", 
                         fontsize=9)
        axes[i].axis('off')
    
    plt.suptitle('Misclassified Samples', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Run Error Analysis
print("="*60)
print("OPTIONAL F: ERROR ANALYSIS")
print("="*60)

# Train a model for error analysis (or use existing trained model)
train_loader, val_loader, train_dataset, val_dataset = get_data_loaders(img_size=IMG_SIZE)

print("\n--- Training model for error analysis ---")
model_for_analysis = DeepCNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE)

config = {
    'experiment': 'F_error_analysis',
    'model_name': 'DeepCNN_for_analysis',
    'epochs': 20,
    'lr': 0.001,
    'optimizer': 'adam',
    'batch_size': BATCH_SIZE,
    'img_size': IMG_SIZE
}

history = train_model(model_for_analysis, train_loader, val_loader, config)

# Get predictions
print("\n--- Getting predictions on validation set ---")
preds, labels, probs = get_predictions(model_for_analysis, val_loader, device)

In [ ]:
# Plot confusion matrix
print("\n--- Confusion Matrix ---")
cm = plot_confusion_matrix(labels, preds, CLASS_NAMES)

# Print classification report
print("\n--- Classification Report ---")
print(classification_report(labels, preds, target_names=CLASS_NAMES))

# Calculate per-class accuracy
print("\n--- Per-Class Accuracy ---")
for i, class_name in enumerate(CLASS_NAMES):
    class_mask = labels == i
    class_acc = (preds[class_mask] == labels[class_mask]).mean() * 100
    print(f"{class_name:12}: {class_acc:.1f}%")

In [ ]:
# Visualize misclassified samples
print("\n--- Misclassified Samples ---")
misclassified = get_misclassified_samples(val_dataset, preds, labels)
print(f"Total misclassified: {len([m for m in range(len(preds)) if preds[m] != labels[m]])} out of {len(preds)}")

show_misclassified(val_dataset, misclassified, CLASS_NAMES, n=10)

# Analyze most common confusions
print("\n--- Most Common Confusions ---")
confusion_pairs = {}
for m in misclassified:
    pair = (CLASS_NAMES[m['true_label']], CLASS_NAMES[m['pred_label']])
    confusion_pairs[pair] = confusion_pairs.get(pair, 0) + 1

sorted_pairs = sorted(confusion_pairs.items(), key=lambda x: x[1], reverse=True)[:10]
for (true_cls, pred_cls), count in sorted_pairs:
    print(f"  {true_cls:10} -> {pred_cls:10}: {count} times")

### Error Analysis: Observations

**YOUR OBSERVATIONS HERE:**

**Common Confusion Patterns:**
- [Which classes are often confused with each other?]
- [Are there any patterns (e.g., similar-looking animals)?]

**Potential Reasons for Misclassifications:**
- Similar visual features (e.g., four-legged animals)
- Image quality issues
- Unusual poses or backgrounds
- Potential mislabeling in dataset

**Suggestions for Improvement:**
- More training data for confused classes
- Data augmentation targeting weak points
- Class-weighted loss function

---
## 8. Optional G: ResNet Comparison <a id='8-resnet'></a>

**Optional Objective:** Use advanced architectures (ResNet) and compare their performance with simpler architectures.

### Experiment Plan
1. Load pretrained ResNet18 (transfer learning)
2. Fine-tune on our dataset
3. Compare with our custom DeepCNN
4. Also try training ResNet from scratch

In [ ]:
from torchvision import models

def get_resnet18(num_classes, pretrained=True):
    """Get ResNet18 model, optionally pretrained."""
    if pretrained:
        # Load pretrained weights
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        # Freeze early layers (optional - comment out for full fine-tuning)
        for param in list(model.parameters())[:-20]:
            param.requires_grad = False
    else:
        model = models.resnet18(weights=None)
    
    # Replace final layer for our number of classes
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    
    return model


def count_parameters(model):
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# ResNet Comparison Experiment
print("="*60)
print("OPTIONAL G: RESNET COMPARISON")
print("="*60)

# Use slightly larger images for ResNet (it expects 224, but 128 works well too)
RESNET_IMG_SIZE = 128  # Balance between speed and performance

train_loader, val_loader, _, _ = get_data_loaders(img_size=RESNET_IMG_SIZE)

resnet_histories = []
resnet_labels = []

# Compare model sizes
print("\n--- Model Parameters ---")
deep_cnn = DeepCNN(num_classes=NUM_CLASSES, img_size=RESNET_IMG_SIZE)
resnet_pretrained = get_resnet18(NUM_CLASSES, pretrained=True)
resnet_scratch = get_resnet18(NUM_CLASSES, pretrained=False)

print(f"DeepCNN:              {count_parameters(deep_cnn):,} trainable params")
print(f"ResNet18 (pretrained): {count_parameters(resnet_pretrained):,} trainable params")
print(f"ResNet18 (scratch):    {count_parameters(resnet_scratch):,} trainable params")

In [ ]:
# Train our DeepCNN for comparison
print("\n--- Training DeepCNN ---")
model = DeepCNN(num_classes=NUM_CLASSES, img_size=RESNET_IMG_SIZE)

config = {
    'experiment': 'G_resnet_comparison',
    'model_name': 'DeepCNN_baseline',
    'epochs': 15,
    'lr': 0.001,
    'optimizer': 'adam',
    'batch_size': BATCH_SIZE,
    'img_size': RESNET_IMG_SIZE,
    'architecture': 'custom_deep_cnn'
}

history = train_model(model, train_loader, val_loader, config)
resnet_histories.append(history)
resnet_labels.append("DeepCNN (ours)")

In [ ]:
# Train ResNet18 with pretrained weights (Transfer Learning)
print("\n--- Training ResNet18 (Pretrained / Transfer Learning) ---")
model = get_resnet18(NUM_CLASSES, pretrained=True)

config = {
    'experiment': 'G_resnet_comparison',
    'model_name': 'ResNet18_pretrained',
    'epochs': 15,
    'lr': 0.001,
    'optimizer': 'adam',
    'batch_size': BATCH_SIZE,
    'img_size': RESNET_IMG_SIZE,
    'architecture': 'resnet18',
    'pretrained': True,
    'transfer_learning': True
}

history = train_model(model, train_loader, val_loader, config)
resnet_histories.append(history)
resnet_labels.append("ResNet18 (pretrained)")

In [ ]:
# Train ResNet18 from scratch (no pretrained weights)
print("\n--- Training ResNet18 (From Scratch) ---")
model = get_resnet18(NUM_CLASSES, pretrained=False)

config = {
    'experiment': 'G_resnet_comparison',
    'model_name': 'ResNet18_scratch',
    'epochs': 15,
    'lr': 0.001,
    'optimizer': 'adam',
    'batch_size': BATCH_SIZE,
    'img_size': RESNET_IMG_SIZE,
    'architecture': 'resnet18',
    'pretrained': False,
    'transfer_learning': False
}

history = train_model(model, train_loader, val_loader, config)
resnet_histories.append(history)
resnet_labels.append("ResNet18 (scratch)")

In [ ]:
# Compare all architectures
compare_histories(resnet_histories, resnet_labels, "ResNet vs Custom CNN Comparison")

print("\n" + "="*60)
print("RESNET COMPARISON: SUMMARY")
print("="*60)
for hist, label in zip(resnet_histories, resnet_labels):
    best_val = max(hist['val_acc'])
    final_val = hist['val_acc'][-1]
    print(f"{label:25} | Best Val: {best_val:.2f}% | Final Val: {final_val:.2f}%")

### ResNet Comparison: Analysis

**YOUR OBSERVATIONS HERE:**

**DeepCNN (Custom):**
- [Performance, training time, simplicity]

**ResNet18 (Pretrained / Transfer Learning):**
- [Should achieve best performance due to pretrained features]
- [Faster convergence expected]
- [Benefits from ImageNet knowledge]

**ResNet18 (From Scratch):**
- [Compare with pretrained version]
- [May need more epochs to converge]
- [Shows value of transfer learning]

**Key Insights:**
- Transfer learning significantly accelerates training
- Pretrained models have learned general features (edges, textures) useful for many tasks
- For limited data, transfer learning often outperforms training from scratch

In [ ]:
compare_histories(aug_histories, aug_labels, "Data Augmentation Comparison")

print("\nSUMMARY:")
for hist, label in zip(aug_histories, aug_labels):
    best_val = max(hist['val_acc'])
    print(f"{label:20} | Best Val Acc: {best_val:.2f}%")

### Data Augmentation Analysis

**YOUR OBSERVATIONS HERE:**

- Augmentation creates artificial variety in training data
- Model sees different perspectives (flips, rotations) of same images
- Reduces overfitting by increasing effective dataset size
- Training loss may be higher, but validation accuracy improves

---
## 8. Results Summary and Conclusions <a id='8-conclusions'></a>

### W&B Dashboard

All experiments are logged to Weights & Biases. View interactive plots and compare runs at:

**https://wandb.ai/YOUR_USERNAME/icosimal-cnn**

(Include screenshots from your W&B dashboard here)

### Summary Table

| Experiment | Key Finding | Best Val Acc |
|------------|-------------|-------------|
| A: Depth | Deeper models learn better features | XX.X% |
| B: Hyperparams | LR=0.001, BS=XX optimal | XX.X% |
| C: Fitting | Demonstrated over/underfitting | - |
| D: Regularization | Dropout + WD reduces gap | XX.X% |
| E: Optimizers | Adam converges fastest | XX.X% |
| Optional: Augmentation | Improves generalization | XX.X% |

### Key Conclusions

1. **Depth matters**: Deeper CNNs can learn hierarchical features but need proper regularization

2. **Hyperparameters are critical**: Wrong learning rate can prevent convergence entirely

3. **Overfitting vs Underfitting**: Understanding the trade-off helps model selection

4. **Regularization**: Essential for preventing overfitting, especially with limited data

5. **Optimizer choice**: Adaptive methods (Adam) are generally more robust

### Best Model Configuration

Based on experiments, the recommended configuration is:
- Architecture: [DeepCNN / MediumCNN]
- Optimizer: Adam, LR=0.001
- Regularization: Dropout=0.3, Weight Decay=1e-4
- Data Augmentation: Yes (horizontal flip, rotation, color jitter)
- Image Size: [64x64 for quick experiments / 224x224 for final]

---
## Appendix: Run Best Model at Full Resolution

After finding good hyperparameters at 64x64, run the final model at 224x224.

In [ ]:
# Final training at full resolution (uncomment to run)
# WARNING: This will take longer!

# FINAL_IMG_SIZE = 224
# FINAL_EPOCHS = 30

# print("Training final model at 224x224...")
# train_loader, val_loader, _, _ = get_data_loaders(img_size=FINAL_IMG_SIZE, augment=True)

# model = DeepCNN(num_classes=NUM_CLASSES, img_size=FINAL_IMG_SIZE)

# config = {
#     'experiment': 'Final_model',
#     'model_name': 'DeepCNN_224_final',
#     'epochs': FINAL_EPOCHS,
#     'lr': 0.001,
#     'optimizer': 'adam',
#     'weight_decay': 1e-4,
#     'batch_size': BATCH_SIZE,
#     'img_size': FINAL_IMG_SIZE,
#     'augmentation': True
# }

# history = train_model(model, train_loader, val_loader, config)
# plot_history(history, "Final Model (224x224)")

# # Save the model
# torch.save(model.state_dict(), 'best_model.pth')
# print(f"\nFinal Best Validation Accuracy: {max(history['val_acc']):.2f}%")